# Structure-augmented residue ablation (A\*02:01, 6 peptides)

Does structure add over sequence, isolated cleanly? Ladder:
- **seq** — Atchley peptide + CDR3 sequence (no structure)
- **seq_pepstruct** — structural (groove-frame) peptide instead of Atchley
- **seq_pepstruct_z** — + frozen conditional-flow **V-marginalized** CDR3 body residual z
- **full** — + per-residue CDR3 geometry (pLDDT-gated, not V-marginalized)
- **full_permz** — control: z shuffled within peptide (pose-permutation)

The conditional flow is **pre-fit + frozen** per fold (label-agnostic p(pose|V)). Run on Colab (GPU).

In [ ]:
import os, sys, subprocess
REPO = '/content/tcrpmhc_pose_binding'
REPO_URL = 'github.com/92kunheekim/tcrpmhc_pose_binding.git'
from google.colab import userdata
TOKEN = userdata.get('GITHUB_TOKEN')
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', f'https://x-access-token:{TOKEN}@{REPO_URL}', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
os.chdir(REPO); sys.path.insert(0, f'{REPO}/src')
DATA_DIR = f'{REPO}/data'; PRETRAIN_DIR = f'{REPO}/pretrained'
RESULTS = f'{REPO}/results'; os.makedirs(RESULTS, exist_ok=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO}/requirements.txt'])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers'])
import json, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from data import load_data, repeated_splits, cluster_tcrs_tcrdist
from train_utils import make_warm_start, aggregate_oof, cluster_bootstrap, DEVICE
from pan_specific import compute_esm_embeddings
from struct_residue import run_struct_ablation
print('device:', DEVICE)

## Config

In [ ]:
ALLELE = 'HLA-A*02:01'
EPOCHS = 20; FLOW_EPOCHS = 150; N_REPEATS = 10; ESM_MAXLEN = 190

## Data + residue features + warm-start + single-allele ESM

In [ ]:
D = load_data(f'{DATA_DIR}/10x_cd8_A0201_6pep', min_iptm=0.5)
pool, trans_scale = D['pool'], D['trans_scale']
res_df = pd.read_csv(f'{DATA_DIR}/10x_cd8_A0201_6pep/residue_frame_features.csv.gz')
print('rows:', len(pool), '| residue-feature ids:', res_df.id.nunique())

a1a2 = json.load(open(f'{DATA_DIR}/pmhc_classI_pretraining/processed/mhc_classI_a1a2.json'))
pool['mhc_seq'] = a1a2[ALLELE]
esm_table = compute_esm_embeddings([a1a2[ALLELE]], maxlen=ESM_MAXLEN)

TCR_ENC   = torch.load(f'{PRETRAIN_DIR}/tcr_encoders_corpus.pt', map_location='cpu')
pep_state = torch.load(f'{PRETRAIN_DIR}/pep_encoder_masked.pt', map_location='cpu')
mhc_state = torch.load(f'{PRETRAIN_DIR}/mhc_encoder_masked.pt', map_location='cpu')
warm = make_warm_start(TCR_ENC, pep_state=pep_state, mhc_state=mhc_state)

## Splits (edit-distance TCR clusters) + run ablation

In [ ]:
groups_ed = cluster_tcrs_tcrdist(pool, ident=0.8)
clu = repeated_splits(pool, n_repeats=N_REPEATS, groups=groups_ed)
print('clusters:', len(np.unique(groups_ed)), '| cluster splits:', len(clu))

metrics, oof = run_struct_ablation(pool, res_df, trans_scale, esm_table, clu,
                                  warm=warm, tcr_enc=TCR_ENC, epochs=EPOCHS,
                                  flow_epochs=FLOW_EPOCHS, permute_z=True)
metrics.to_csv(f'{RESULTS}/struct_ablation_per_split.csv', index=False)
summary = metrics.groupby('config')[['auroc','auprc']].agg(['mean','std']).round(3)
print(summary); summary.to_csv(f'{RESULTS}/struct_ablation_summary.csv')

## Paired cluster-bootstrap contrasts (per-sample aggregated)

Each Δ isolates one addition; positive CI excluding 0 = that block adds signal.
`full - full_permz` is the pose-permutation check (should be ~0 if z is inert).

In [ ]:
y = pool.label.to_numpy().astype(int); n = len(pool)
def agg(c):
    pa, tested = aggregate_oof(oof[c]['idx'], oof[c]['p'], n); return pa, tested
pairs = [('seq_pepstruct','seq'), ('seq_pepstruct_z','seq_pepstruct'),
         ('full','seq_pepstruct_z'), ('full','full_permz')]
for a, b in pairs:
    pa, ta = agg(a); pb, tb = agg(b); tested = ta & tb
    (dR, loR, hiR, pR), (dP, loP, hiP, pP) = cluster_bootstrap(y, pa, pb, groups_ed, tested)
    print(f'{a:16s} - {b:16s}:  dAUROC={dR:+.3f} [{loR:+.3f},{hiR:+.3f}] p={pR:.3f}   '
          f'dAUPRC={dP:+.3f} [{loP:+.3f},{hiP:+.3f}]')

## Commit & push

In [ ]:
import json
for _nb in ['notebooks/ablation_struct.ipynb']:
    _p = f'{REPO}/{_nb}'
    if os.path.exists(_p):
        _d = json.load(open(_p)); _d.get('metadata', {}).pop('widgets', None)
        for _c in _d.get('cells', []): _c.get('metadata', {}).pop('widgets', None)
        json.dump(_d, open(_p, 'w'), indent=1)
subprocess.run(['git', 'config', 'user.email', '92kunheekim@gmail.com'], cwd=REPO)
subprocess.run(['git', 'config', 'user.name', 'KH Kim'], cwd=REPO)
subprocess.run(['git', 'add', '-f', 'results', 'notebooks/ablation_struct.ipynb'], cwd=REPO)
subprocess.run(['git', 'commit', '-m', 'Structure-augmented residue ablation results [colab]'], cwd=REPO)
subprocess.run(['git', 'push', 'origin', 'main'], cwd=REPO, check=True)
print('pushed.')